# 🧪 Brouillon 01 — CNN simple (baseline)

**Projet :** Ultra Pro Access Core AI — Safe Exit Pro  
**Équipe :** Yassine Mokni & Hadil Dhaya  

## Objectif de l'expérimentation

Ce carnet documente une **première approche naïve** : entraîner un **CNN convolutionnel léger from scratch** sur des crops de visages pour classifier les identités.

> ⚠️ **Statut : brouillon / rejeté** — précision insuffisante pour un contrôle d'accès en conditions réelles.

## Hypothèse initiale

Un réseau peu profond pourrait suffire si le nombre d'utilisateurs reste faible et si l'éclairage est stable.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'drafts' else Path.cwd().resolve()
if (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT if (PROJECT_ROOT / 'dataset').exists() else PROJECT_ROOT.parent
else:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / 'dataset'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_DIR exists:', DATASET_DIR.exists())

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# CNN baseline très simple (draft)
class TinyFaceCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.flatten(1))

transform = transforms.Compose([
    transforms.Resize((64, 64)),  # résolution basse → perte d'information
    transforms.ToTensor(),
])

if DATASET_DIR.exists():
    ds = datasets.ImageFolder(str(DATASET_DIR), transform=transform)
    loader = DataLoader(ds, batch_size=16, shuffle=True)
    model = TinyFaceCNN(num_classes=len(ds.classes))
    print('Classes:', ds.classes)
    print('Échantillons:', len(ds))
else:
    print('⚠️ dataset/ introuvable — exécuter depuis la racine du projet.')

In [ ]:
# Entraînement court de démonstration (brouillon)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) if DATASET_DIR.exists() else None

if DATASET_DIR.exists():
    model.train()
    for epoch in range(3):
        total_loss = 0.0
        for images, labels in loader:
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1} — loss moyenne: {total_loss/len(loader):.4f}')

## Conclusion du brouillon

| Limite observée | Impact |
|-----------------|--------|
| Peu de données par classe | Sur-apprentissage rapide |
| CNN shallow | Features génériques faibles |
| Résolution 64×64 | Traits biométriques perdus |

➡️ **Décision :** abandon du CNN custom au profit d'**embeddings FaceNet** (InceptionResnetV1 pré-entraîné VGGFace2).